---

Import libraries and define directories path

---

In [1]:
import os
import json
import shutil
import zipfile, tempfile, shutil
import logging
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import yaml

from PIL import Image
import cv2 as cv

from tqdm import tqdm
import matplotlib.pyplot as plt

from cvat_sdk import make_client
from cvat_sdk.core.proxies.tasks import ResourceType

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

VALIDATED_DIR = PROJECT_ROOT / "datasets" / "validated"
INTERIM_DIR   = PROJECT_ROOT / "datasets" / "interim"
REPORT_DIR    = PROJECT_ROOT / "datasets" / "reports"
CONFIG_PATH   = PROJECT_ROOT / "config" / "download_sheet.yaml"

# Sanity check the input actually exists before we go further
assert VALIDATED_DIR.exists(), f"validated/ not found at {VALIDATED_DIR}"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"VALIDATED_DIR  = {VALIDATED_DIR}")
print(f"INTERIM_DIR    = {INTERIM_DIR}")

PROJECT_ROOT   = D:\SIT374\WalkBuddy-T2-2026\ML_side
VALIDATED_DIR  = D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated
INTERIM_DIR    = D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim


---

Set up log

---

In [3]:
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_PATH = REPORT_DIR / f"annotation_review_log_{RUN_TIMESTAMP}.log"

logger = logging.getLogger("annotation_review")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

# Console handler
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(console_handler)

# File handler
file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

logger.info("=== Annotation Review Notebook Started ===")
logger.info(f"Run timestamp: {RUN_TIMESTAMP}")
logger.info(f"VALIDATED_DIR: {VALIDATED_DIR}")
logger.info(f"INTERIM_DIR: {INTERIM_DIR}")
logger.info(f"Log file: {LOG_PATH}")

print(f"Logging to: {LOG_PATH}")

2026-08-05 15:24:34,212 | INFO | === Annotation Review Notebook Started ===
2026-08-05 15:24:34,213 | INFO | Run timestamp: 20260805_152434
2026-08-05 15:24:34,214 | INFO | VALIDATED_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated
2026-08-05 15:24:34,215 | INFO | INTERIM_DIR: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim
2026-08-05 15:24:34,216 | INFO | Log file: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\annotation_review_log_20260805_152434.log


Logging to: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\annotation_review_log_20260805_152434.log


---

Read the yaml AGAIN, and confirm the structure of validated

---

In [4]:
# --- Load YAML config ---
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

datasets_config = config["datasets"] if "datasets" in config else config
approved_datasets = {
    name: meta for name, meta in datasets_config.items()
    if meta.get("status", "approved") == "approved"
}

logger.info(f"Loaded config: {len(approved_datasets)} approved datasets")

# --- Inventory validated/ (source/dataset_name/... structure) ---
inventory = []

for source_dir in sorted(VALIDATED_DIR.iterdir()):
    if not source_dir.is_dir():
        continue
    for dataset_dir in sorted(source_dir.iterdir()):
        if not dataset_dir.is_dir():
            continue

        file_count = sum(1 for _ in dataset_dir.rglob("*") if _.is_file())
        dataset_name = dataset_dir.name

        inventory.append({
            "source": source_dir.name,
            "dataset_name": dataset_name,
            "path": str(dataset_dir),
            "file_count": file_count,
            "in_config": dataset_name in approved_datasets,
        })

inventory_df = pd.DataFrame(inventory)

logger.info(f"Inventoried {len(inventory_df)} dataset folders under validated/")
logger.info(f"Total files across validated/: {inventory_df['file_count'].sum()}")

# Flag anything in validated/ that isn't in the YAML (shouldn't happen, but worth catching)
unlisted = inventory_df[~inventory_df["in_config"]]
if len(unlisted) > 0:
    logger.warning(f"{len(unlisted)} dataset folder(s) in validated/ not found in YAML config:")
    for _, row in unlisted.iterrows():
        logger.warning(f"  - {row['source']}/{row['dataset_name']}")

inventory_df

2026-08-05 04:38:34,731 | INFO | Loaded config: 18 approved datasets
2026-08-05 04:38:53,583 | INFO | Inventoried 18 dataset folders under validated/
2026-08-05 04:38:53,588 | INFO | Total files across validated/: 277505


,source,dataset_name,path,file_count,in_config
0,kaggle,antic_chairs,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,1403,True
1,kaggle,doors_detection,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,208,True
2,kaggle,fpv_crosswalk_segmentation,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,6600,True
3,kaggle,indoor_object_detection,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,2533,True
4,kaggle,light_poles,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,1819,True
5,kaggle,obstacle_detection_kaggle,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,47932,True
6,kaggle,pedestrian_detection,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,6,True
7,kaggle,road_sign_detection,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,1754,True
8,manual,mapillary_vistas,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,165008,True
9,roboflow,footpath_detection,D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\v...,3225,True


---

Inspect each dataset anotation type

---

In [5]:
def detect_format(dataset_dir: Path) -> str:
    files = list(dataset_dir.rglob("*"))
    suffixes = Counter(f.suffix.lower() for f in files if f.is_file())

    if dataset_dir.name == "mapillary_vistas":
        return "mapillary_panoptic_json"

    has_txt = suffixes.get(".txt", 0) > 0
    has_xml = suffixes.get(".xml", 0) > 0
    has_json = suffixes.get(".json", 0) > 0
    has_images = any(suffixes.get(ext, 0) > 0 for ext in [".jpg", ".jpeg", ".png"])

    # NEW: check for paired _mask images before assuming image_only
    image_files = [f for f in files if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    mask_stems = {f.stem for f in image_files if f.stem.endswith("_mask")}
    non_mask_stems = {f.stem for f in image_files if not f.stem.endswith("_mask")}
    has_mask_pairs = len(mask_stems) > 0 and any(f"{s}_mask" in mask_stems for s in non_mask_stems)

    if has_mask_pairs:
        return "segmentation_mask_pairs"
    elif has_xml:
        return "pascal_voc"
    elif has_json and not has_txt:
        return "coco"
    elif has_txt:
        return "yolo"
    elif has_images and not (has_txt or has_xml or has_json):
        return "image_only"
    else:
        return "unknown"

format_results = []
for _, row in inventory_df.iterrows():
    fmt = detect_format(Path(row["path"]))
    format_results.append(fmt)
    logger.info(f"{row['source']}/{row['dataset_name']}: detected format = {fmt}")

inventory_df["annotation_format"] = format_results

# Surface anything ambiguous for manual review before we go further
unknown = inventory_df[inventory_df["annotation_format"] == "unknown"]
if len(unknown) > 0:
    logger.warning(f"{len(unknown)} dataset(s) with undetected format — needs manual check:")
    for _, row in unknown.iterrows():
        logger.warning(f"  - {row['source']}/{row['dataset_name']}")

inventory_df[["source", "dataset_name", "file_count", "annotation_format"]]

2026-08-05 04:38:53,904 | INFO | kaggle/antic_chairs: detected format = image_only
2026-08-05 04:38:53,937 | INFO | kaggle/doors_detection: detected format = image_only
2026-08-05 04:38:54,992 | INFO | kaggle/fpv_crosswalk_segmentation: detected format = segmentation_mask_pairs
2026-08-05 04:38:55,331 | INFO | kaggle/indoor_object_detection: detected format = yolo
2026-08-05 04:38:55,613 | INFO | kaggle/light_poles: detected format = yolo
2026-08-05 04:39:02,997 | INFO | kaggle/obstacle_detection_kaggle: detected format = yolo
2026-08-05 04:39:03,002 | INFO | kaggle/pedestrian_detection: detected format = unknown
2026-08-05 04:39:03,217 | INFO | kaggle/road_sign_detection: detected format = pascal_voc
2026-08-05 04:39:21,234 | INFO | manual/mapillary_vistas: detected format = mapillary_panoptic_json
2026-08-05 04:39:22,226 | INFO | roboflow/footpath_detection: detected format = yolo
2026-08-05 04:39:23,420 | INFO | roboflow/indoor_detection_vineeth: detected format = yolo
2026-08-05 04

,source,dataset_name,file_count,annotation_format
0,kaggle,antic_chairs,1403,image_only
1,kaggle,doors_detection,208,image_only
2,kaggle,fpv_crosswalk_segmentation,6600,segmentation_mask_pairs
3,kaggle,indoor_object_detection,2533,yolo
4,kaggle,light_poles,1819,yolo
5,kaggle,obstacle_detection_kaggle,47932,yolo
6,kaggle,pedestrian_detection,6,unknown
7,kaggle,road_sign_detection,1754,pascal_voc
8,manual,mapillary_vistas,165008,mapillary_panoptic_json
9,roboflow,footpath_detection,3225,yolo


---

We'll start with the dataset with video, which is pedestrian detection (this goes straight to interim compared to other dataset as it has all bounding boxes info for each frame and there is only one object (you could write more check if you want)

---

In [7]:
DATASET_DIR = VALIDATED_DIR / "kaggle" / "pedestrian_detection"
PEDESTRIAN_OUT_DIR = INTERIM_DIR / "kaggle" / "pedestrian_detection"
PEDESTRIAN_OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ID = 0  # single class: pedestrian
video_pairs = ["crosswalk", "fourway", "night"]
conversion_log = []

for name in video_pairs:
    video_path = DATASET_DIR / f"{name}.avi"
    csv_path = DATASET_DIR / f"{name}.csv"
    df = pd.read_csv(csv_path)

    cap = cv.VideoCapture(str(video_path))
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        img_h, img_w = frame.shape[:2]
        row = df.iloc[frame_idx]
        x, y, w, h = row["x"], row["y"], row["w"], row["h"]

        # Convert top-left x,y,w,h -> normalized YOLO center format
        x_center = (x + w / 2) / img_w
        y_center = (y + h / 2) / img_h
        w_norm = w / img_w
        h_norm = h / img_h

        # Filenames: prefix with source video name to avoid collisions across the 3 pairs
        stem = f"{name}_{frame_idx:05d}"
        img_out = PEDESTRIAN_OUT_DIR / f"{stem}.jpg"
        label_out = PEDESTRIAN_OUT_DIR / f"{stem}.txt"

        cv.imwrite(str(img_out), frame)
        with open(label_out, "w") as f:
            f.write(f"{CLASS_ID} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

        frame_idx += 1

    cap.release()
    conversion_log.append({"video": name, "frames_extracted": frame_idx})
    logger.info(f"{name}: extracted {frame_idx} frames + YOLO labels -> {PEDESTRIAN_OUT_DIR}")

pd.DataFrame(conversion_log)

2026-08-05 04:44:18,206 | INFO | crosswalk: extracted 378 frames + YOLO labels -> D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\kaggle\pedestrian_detection
2026-08-05 04:46:55,238 | INFO | fourway: extracted 1281 frames + YOLO labels -> D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\kaggle\pedestrian_detection
2026-08-05 04:47:47,662 | INFO | night: extracted 565 frames + YOLO labels -> D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\kaggle\pedestrian_detection


,video,frames_extracted
0,crosswalk,378
1,fourway,1281
2,night,565


---

Setting up the CVAT, you'll need docker and cvat installed in order to run this

---

In [13]:
CVAT_HOST = os.environ.get("CVAT_HOST")
CVAT_USER = os.environ.get("CVAT_USER")
CVAT_PASS = os.environ.get("CVAT_PASS")

assert CVAT_HOST and CVAT_USER and CVAT_PASS, (
    "Missing CVAT_HOST/CVAT_USER/CVAT_PASS environment variables — "
    "did you restart the kernel after setting them?"
)

client = make_client(host=CVAT_HOST, credentials=(CVAT_USER, CVAT_PASS))

# Quick connectivity check
tasks = list(client.tasks.list())
logger.info(f"Connected to CVAT at {CVAT_HOST} as {CVAT_USER}")
logger.info(f"Existing tasks on server: {len(tasks)}")

print(f"Connected to CVAT at {CVAT_HOST} — {len(tasks)} existing task(s)")

2026-08-04 19:14:23,726 | INFO | Connected to CVAT at http://localhost:8081 as min
2026-08-04 19:14:23,729 | INFO | Existing tasks on server: 0


Connected to CVAT at http://localhost:8081 — 0 existing task(s)


---

Group dataset by sample for the review

---

In [45]:
SAMPLE_SIZE = 100   # was 150
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

EXCLUDED_DATASETS = {
    "mapillary_vistas",
    "pedestrian_detection",
    "antic_chairs",
    "doors_detection",
    "fpv_crosswalk_segmentation",
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

sample_records = []

for _, row in inventory_df.iterrows():
    dataset_name = row["dataset_name"]
    if dataset_name in EXCLUDED_DATASETS:
        logger.info(f"Skipping {dataset_name} (handled separately)")
        continue

    dataset_dir = Path(row["path"])
    fmt = row["annotation_format"]

    all_images = sorted(f for f in dataset_dir.rglob("*") if f.suffix.lower() in IMAGE_EXTS)
    n_available = len(all_images)
    n_to_sample = min(SAMPLE_SIZE, n_available)
    if n_to_sample == 0:
        logger.warning(f"{dataset_name}: no images found")
        continue

    sampled = list(np.random.choice(all_images, size=n_to_sample, replace=False))

    label_lookup = {}
    if fmt == "yolo":
        for f in dataset_dir.rglob("*.txt"):
            label_lookup[f.stem] = f
    elif fmt == "pascal_voc":
        for f in dataset_dir.rglob("*.xml"):
            label_lookup[f.stem] = f

    matched, unmatched = 0, 0
    for img_path in sampled:
        label_path = label_lookup.get(img_path.stem)
        if fmt in {"yolo", "pascal_voc"}:
            matched += label_path is not None
            unmatched += label_path is None
        sample_records.append({
            "source": row["source"],
            "dataset_name": dataset_name,
            "annotation_format": fmt,
            "image_path": str(img_path),
            "label_path": str(label_path) if label_path else None,
        })

    if fmt in {"yolo", "pascal_voc"}:
        logger.info(f"{dataset_name}: sampled {n_to_sample}/{n_available} — {matched} matched, {unmatched} unmatched")
    else:
        logger.info(f"{dataset_name}: sampled {n_to_sample}/{n_available} (no per-image labels expected)")

sample_df = pd.DataFrame(sample_records)
logger.info(f"Total sampled images across all datasets: {len(sample_df)}")

sample_df.groupby(["dataset_name", "annotation_format"])["label_path"].apply(
    lambda x: f"{x.notna().sum()}/{len(x)} matched"
)

2026-08-04 22:07:02,539 | INFO | Skipping antic_chairs (handled separately)
2026-08-04 22:07:02,542 | INFO | Skipping doors_detection (handled separately)
2026-08-04 22:07:02,545 | INFO | Skipping fpv_crosswalk_segmentation (handled separately)
2026-08-04 22:07:02,759 | INFO | indoor_object_detection: sampled 100/1340 — 89 matched, 11 unmatched
2026-08-04 22:07:02,894 | INFO | light_poles: sampled 100/910 — 100 matched, 0 unmatched
2026-08-04 22:07:07,116 | INFO | obstacle_detection_kaggle: sampled 100/24249 — 98 matched, 2 unmatched
2026-08-04 22:07:07,119 | INFO | Skipping pedestrian_detection (handled separately)
2026-08-04 22:07:07,319 | INFO | road_sign_detection: sampled 100/877 — 100 matched, 0 unmatched
2026-08-04 22:07:07,323 | INFO | Skipping mapillary_vistas (handled separately)
2026-08-04 22:07:07,636 | INFO | footpath_detection: sampled 100/1687 — 93 matched, 7 unmatched
2026-08-04 22:07:08,059 | INFO | indoor_detection_vineeth: sampled 100/2890 — 100 matched, 0 unmatched


dataset_name                           annotation_format
footpath_detection                     yolo                  93/100 matched
indoor_detection_vineeth               yolo                 100/100 matched
indoor_object_detection                yolo                  89/100 matched
indoor_objects_5iwhq                   yolo                 100/100 matched
indoor_objects_roboflow                yolo                 100/100 matched
light_poles                            yolo                 100/100 matched
obstacle_detection_kaggle              yolo                  98/100 matched
obstacle_detection_roboflow            yolo                  95/100 matched
outdoor_objects                        yolo                  98/100 matched
pedestrian_walk                        yolo                 100/100 matched
revised_pedestrian_obstacle_detection  yolo                 100/100 matched
road_sign_detection                    pascal_voc           100/100 matched
stairs_detection               

---

Create CVAT tasks per dataset

---

In [46]:
task_ids = {}

for dataset_name, group in sample_df.groupby("dataset_name"):
    image_paths = group["image_path"].tolist()
    classes = dataset_classes.get(dataset_name)

    if not classes:
        logger.warning(f"{dataset_name}: no classes resolved — skipping task creation")
        continue

    task_spec = {
        "name": f"walkbuddy_{dataset_name}_review",
        "labels": [{"name": c} for c in classes],
    }

    task = client.tasks.create_from_data(
        spec=task_spec,
        resources=image_paths,
        resource_type=ResourceType.LOCAL,
    )

    task_ids[dataset_name] = task.id
    logger.info(f"{dataset_name}: created task id={task.id}, {len(image_paths)} images, {len(classes)} labels")

print(f"Created {len(task_ids)} CVAT tasks")
task_ids

2026-08-04 22:11:23,923 | INFO | footpath_detection: created task id=10, 100 images, 5 labels
2026-08-04 22:11:29,444 | INFO | indoor_detection_vineeth: created task id=11, 100 images, 10 labels
2026-08-04 22:11:35,577 | INFO | indoor_object_detection: created task id=12, 100 images, 10 labels
2026-08-04 22:11:41,333 | INFO | indoor_objects_5iwhq: created task id=13, 100 images, 10 labels
2026-08-04 22:11:48,703 | INFO | indoor_objects_roboflow: created task id=14, 100 images, 125 labels
2026-08-04 22:11:54,454 | INFO | light_poles: created task id=15, 100 images, 1 labels
2026-08-04 22:12:00,293 | INFO | obstacle_detection_kaggle: created task id=16, 100 images, 25 labels
2026-08-04 22:12:06,641 | INFO | obstacle_detection_roboflow: created task id=17, 100 images, 10 labels
2026-08-04 22:12:12,296 | INFO | outdoor_objects: created task id=18, 100 images, 7 labels
2026-08-04 22:12:19,792 | INFO | pedestrian_walk: created task id=19, 100 images, 3 labels
2026-08-04 22:12:27,482 | INFO |

Created 13 CVAT tasks


{'footpath_detection': 10,
 'indoor_detection_vineeth': 11,
 'indoor_object_detection': 12,
 'indoor_objects_5iwhq': 13,
 'indoor_objects_roboflow': 14,
 'light_poles': 15,
 'obstacle_detection_kaggle': 16,
 'obstacle_detection_roboflow': 17,
 'outdoor_objects': 18,
 'pedestrian_walk': 19,
 'revised_pedestrian_obstacle_detection': 20,
 'road_sign_detection': 21,
 'stairs_detection': 22}

---

Upload it for annotation review

---

In [50]:
for dataset_name, task_id in task_ids.items():
    print(f"\n== {dataset_name} ==")
    group = sample_df[sample_df["dataset_name"] == dataset_name]
    if group.empty:
        continue

    fmt = group["annotation_format"].iloc[0]
    if fmt not in {"yolo", "pascal_voc"}:
        logger.info(f"{dataset_name}: no annotations to import (format={fmt})")
        continue

    task = client.tasks.retrieve(task_id)

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)

        if fmt == "yolo":
            classes = dataset_classes[dataset_name]
            obj_dir = tmp_path / "obj_train_data"
            obj_dir.mkdir()

            train_lines, invalid_labels = [], []
            for _, row in group.iterrows():
                if row["label_path"] is None:
                    continue
                img_src, label_src = Path(row["image_path"]), Path(row["label_path"])
                if not img_src.exists() or not label_src.exists():
                    continue

                # validate: 5 fields per line, class_id in range
                lines = [l for l in label_src.read_text(encoding="utf-8").splitlines() if l.strip()]
                valid = all(
                    len(l.split()) == 5 and l.split()[0].isdigit() and int(l.split()[0]) < len(classes)
                    for l in lines
                )
                if not valid:
                    invalid_labels.append(label_src.name)
                    continue

                shutil.copy2(img_src, obj_dir / img_src.name)
                shutil.copy2(label_src, obj_dir / label_src.name)
                train_lines.append(f"obj_train_data/{img_src.name}")

            if invalid_labels:
                print(f"  skipped {len(invalid_labels)} invalid labels: {invalid_labels[:5]}")
            if not train_lines:
                print(f"  no valid annotations, skipping")
                continue

            (tmp_path / "obj.names").write_text("\n".join(classes), encoding="utf-8")
            (tmp_path / "train.txt").write_text("\n".join(train_lines), encoding="utf-8")
            (tmp_path / "obj.data").write_text(
                f"classes = {len(classes)}\nnames = obj.names\ntrain = train.txt\n", encoding="utf-8"
            )

            zip_path = tmp_path / "annotations.zip"
            with zipfile.ZipFile(zip_path, "w") as zf:
                for name in ["obj.data", "obj.names", "train.txt"]:
                    zf.write(tmp_path / name, name)
                for f in obj_dir.iterdir():
                    zf.write(f, f"obj_train_data/{f.name}")

            task.import_annotations(format_name="YOLO 1.1", filename=str(zip_path))
            logger.info(f"{dataset_name}: imported {len(train_lines)} YOLO annotation files")

        elif fmt == "pascal_voc":
            ann_dir, img_dir, sets_dir = tmp_path / "Annotations", tmp_path / "JPEGImages", tmp_path / "ImageSets" / "Main"
            for d in (ann_dir, img_dir, sets_dir):
                d.mkdir(parents=True)

            train_files = []
            for _, row in group.iterrows():
                if row["label_path"] is None:
                    continue
                xml_src, img_src = Path(row["label_path"]), Path(row["image_path"])
                if not xml_src.exists() or not img_src.exists():
                    continue
                shutil.copy2(xml_src, ann_dir / xml_src.name)
                shutil.copy2(img_src, img_dir / img_src.name)
                train_files.append(img_src.stem)

            if not train_files:
                print(f"  no valid VOC annotations, skipping")
                continue

            (sets_dir / "train.txt").write_text("\n".join(train_files), encoding="utf-8")

            zip_path = tmp_path / "annotations.zip"
            with zipfile.ZipFile(zip_path, "w") as zf:
                for f in ann_dir.iterdir():
                    zf.write(f, f"Annotations/{f.name}")
                for f in img_dir.iterdir():
                    zf.write(f, f"JPEGImages/{f.name}")
                zf.write(sets_dir / "train.txt", "ImageSets/Main/train.txt")

            task.import_annotations(format_name="PASCAL VOC 1.1", filename=str(zip_path))
            logger.info(f"{dataset_name}: imported {len(train_files)} Pascal VOC annotation files")

print("\nAnnotation import complete")


========== Processing: footpath_detection ==========
footpath_detection: preparing YOLO annotations
footpath_detection: uploading 93 YOLO annotations


2026-08-04 22:29:47,486 | INFO | footpath_detection: imported 93 YOLO annotation files



========== Processing: indoor_detection_vineeth ==========
indoor_detection_vineeth: preparing YOLO annotations
indoor_detection_vineeth: uploading 100 YOLO annotations


2026-08-04 22:29:56,203 | INFO | indoor_detection_vineeth: imported 100 YOLO annotation files



========== Processing: indoor_object_detection ==========
indoor_object_detection: preparing YOLO annotations
indoor_object_detection: uploading 89 YOLO annotations


...


========== Processing: road_sign_detection ==========
road_sign_detection: preparing Pascal VOC annotations
road_sign_detection: uploading 100 Pascal VOC annotations


2026-08-04 22:31:20,137 | INFO | road_sign_detection: imported 100 Pascal VOC annotation files



========== Processing: stairs_detection ==========
stairs_detection: preparing YOLO annotations
stairs_detection: uploading 98 YOLO annotations


2026-08-04 22:31:30,382 | INFO | stairs_detection: imported 98 YOLO annotation files



Annotation import complete for all applicable datasets


---

Export the annotated samples back (hope you had some funny looking over them or not)

---

In [51]:
reviewed_manifest = []

for dataset_name, task_id in task_ids.items():
    row = inventory_df.loc[inventory_df["dataset_name"] == dataset_name].iloc[0]
    src_dir = Path(row["path"])
    interim_dataset_dir = INTERIM_DIR / row["source"] / dataset_name

    # Step 1: full mirror of validated/ -> interim/ (rebuilt fresh, same convention as validated/invalid)
    if interim_dataset_dir.exists():
        shutil.rmtree(interim_dataset_dir)
    shutil.copytree(src_dir, interim_dataset_dir)
    logger.info(f"{dataset_name}: mirrored to interim/ ({sum(1 for _ in interim_dataset_dir.rglob('*') if _.is_file())} files)")

    # Step 2: export corrected annotations from CVAT, overwrite only reviewed labels
    group = sample_df[sample_df["dataset_name"] == dataset_name]
    fmt = group["annotation_format"].iloc[0]
    if fmt not in {"yolo", "pascal_voc"}:
        continue

    task = client.tasks.retrieve(task_id)
    format_name = "YOLO 1.1" if fmt == "yolo" else "PASCAL VOC 1.1"

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        export_zip = tmp_path / "export.zip"
        task.export_dataset(format_name=format_name, filename=str(export_zip))

        extracted = tmp_path / "extracted"
        with zipfile.ZipFile(export_zip) as zf:
            zf.extractall(extracted)

        exported_dir = extracted / ("obj_train_data" if fmt == "yolo" else "Annotations")
        ext = ".txt" if fmt == "yolo" else ".xml"

        overwritten = 0
        for _, srow in group.iterrows():
            img_stem = Path(srow["image_path"]).stem
            exported_label = exported_dir / f"{img_stem}{ext}"
            if not exported_label.exists():
                continue

            # map back to the equivalent path under interim/
            if srow["label_path"]:
                rel = Path(srow["label_path"]).relative_to(VALIDATED_DIR)
                dest_label_path = INTERIM_DIR / rel
            else:
                dest_label_path = interim_dataset_dir / f"{img_stem}{ext}"

            dest_label_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(exported_label, dest_label_path)
            overwritten += 1

            reviewed_manifest.append({
                "dataset_name": dataset_name,
                "image": img_stem,
                "label_path": str(dest_label_path),
            })

    logger.info(f"{dataset_name}: overwrote {overwritten} reviewed labels in interim/")

manifest_df = pd.DataFrame(reviewed_manifest)
manifest_path = REPORT_DIR / f"reviewed_manifest_{RUN_TIMESTAMP}.csv"
manifest_df.to_csv(manifest_path, index=False)
logger.info(f"Reviewed manifest: {len(manifest_df)} entries saved to {manifest_path}")

print(f"interim/ populated for {len(task_ids)} datasets — {len(manifest_df)} reviewed labels applied")
print(f"Manifest: {manifest_path}")

2026-08-05 03:18:28,562 | INFO | footpath_detection: mirrored to interim/ (3225 files)
2026-08-05 03:18:34,921 | INFO | footpath_detection: overwrote 100 reviewed labels in interim/
2026-08-05 03:18:45,783 | INFO | indoor_detection_vineeth: mirrored to interim/ (5779 files)
2026-08-05 03:18:51,913 | INFO | indoor_detection_vineeth: overwrote 100 reviewed labels in interim/
2026-08-05 03:18:56,828 | INFO | indoor_object_detection: mirrored to interim/ (2533 files)
2026-08-05 03:19:03,115 | INFO | indoor_object_detection: overwrote 100 reviewed labels in interim/
2026-08-05 03:19:07,897 | INFO | indoor_objects_5iwhq: mirrored to interim/ (2975 files)
2026-08-05 03:19:13,957 | INFO | indoor_objects_5iwhq: overwrote 100 reviewed labels in interim/
2026-08-05 03:19:14,328 | INFO | indoor_objects_roboflow: mirrored to interim/ (260 files)
2026-08-05 03:19:20,425 | INFO | indoor_objects_roboflow: overwrote 100 reviewed labels in interim/
2026-08-05 03:19:23,485 | INFO | light_poles: mirrored 

interim/ populated for 13 datasets — 1300 reviewed labels applied
Manifest: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\reports\reviewed_manifest_20260804_190108.csv


In [52]:
client.close()